# Introduction
Elections has an importance at any country. For this reason, we did an
exploratory data analysis of Brazilian election data from 2022, retrieved
from Superior Electoral Court (TSE, in Brazilian Portuguese). We performed
univariate and multivariate analysis to describe and investigate
relationships about vote, party and region, which includes a data
dictionary as well. We found the job, mayor or councilor, plays an
important role in the amount of votes received. Moreover, we also found
a positive correlation between the number of mayors an councilors elected
by party.

In Dataset Analysis section we removed duplicated lines, unimportant
columns and treated null values. Then, in Exploratory Data Analysis
section, univariate and multivariate analysis were performed, followed by
correlation analysis between mayors and councilors elected.

# Dataset Analysis
In this section we just examine data frame structure, by checking
its data type, duplicated rows and columns. After this
preliminary analysis, we provide a data dictionary.

## Importing Required Modules

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

In [2]:
# added project root to path in order to import local modules
sys.path.append(str(Path(os.path.abspath('')).resolve().parents[0]))

In [3]:
from brazilian_elections.config import PROCESSED_DATA_DIR
from brazilian_elections.config import RAW_DATA_DIR
from brazilian_elections.config import REPORTS_DIR
from brazilian_elections.config import FIGURES_DIR

2025-03-27 07:28:07.164 | INFO     | brazilian_elections.config:<module>:12 - PROJ_ROOT path is: /home/gasobral/Meus Arquivos/data-science/brazilian_elections


In [6]:
# checking if the project directory structure was loaded properly
print(f"RAW_DATA_DIR path:       {RAW_DATA_DIR}\n"
      f"PROCESSED_DATA_DIR path: {PROCESSED_DATA_DIR}\n")

print(f"REPORTS_DIR path:        {REPORTS_DIR}\n"
      f"FIGURES_DIR path:        {FIGURES_DIR}\n")

RAW_DATA_DIR path:       /home/gasobral/Meus Arquivos/data-science/brazilian_elections/data/raw
PROCESSED_DATA_DIR path: /home/gasobral/Meus Arquivos/data-science/brazilian_elections/data/processed

REPORTS_DIR path:        /home/gasobral/Meus Arquivos/data-science/brazilian_elections/reports
FIGURES_DIR path:        /home/gasobral/Meus Arquivos/data-science/brazilian_elections/reports/figures



## Loading the Data Set

In [ ]:
from brazilian_elections import dataset

In [ ]:
dataset.main()

In [ ]:
data_frame = pd.read_csv( PROCESSED_DATA_DIR / 'clean_data.csv', engine='pyarrow')

## Checking Data Set Structure

In [ ]:
data_frame.info()

In [ ]:
data_frame.head()

In [ ]:
data_frame.duplicated().sum()

By first looking at the data, we can see that only the column
*legend_votes* has some missing values. This happens because some jobs,
like mayor, do not receive legend votes, which are votes given for
the main party instead of candidate. When analysing data for mayors, we
can simply ignore this column. We also checked that there are no
duplicated rows.

We can also note that the frist column just
enumerate the rows of the dataset. Since we already have an index for
enumerating the rows, then it is ok to remove this column.

Moreover, as the columns *type_id*, *codigo_tse*, *codigo_ibge* and
*job_count* refer to some internal codes used by brazilian government
agencies, then we will remove these columns as well.

In [ ]:
data_frame.drop(columns=['', 'type_id', 'codigo_tse', 'codigo_ibge', 'job_count'],
                inplace=True)

In [ ]:
data_frame.columns

In [ ]:
data_frame.info()

After removing the columns, the memory usage for the data frame was
slightly reduced, going from 114.8 Mb to 95Mb.

## Data Dictionary
Before data is analyzed, we provide a data dictionary break down by variable type.

- *Categorical Data*
  - **uf**: acronym of brazilian state;
  - **nome_municipio**: name of the city;
  - **capital**: it is a binary variable which indicates if the ctiy is capital (capital=1) or not (capital=0);
  - **elector_presence**: the presence of ellectors;
  - **candidate_name**: name of the candidate;
  - **candidate_coligation**: coalition (parties) at which a candidadate belogns to;
  - **candidate_vice_name**: name of the vice candidate;
  - **candidate_vote_destination**: this column tells if the vote is valid, nulled by justice, null or voted a party (instead of canidate);
  - **job**: the job a candidate applied for, mayor or councilor;
  - **main_party**: the party that a candidate belongs to;
- *Numerical Data*
  - **elector_presence**: number of voters of a polling station;
  - **absentees**: number of voters that did not vote;
  - **nominal_votes**: number of votes given for a candidate;
  - **blank_votes**: number of blank votes;
  - **total_null_votes**: total number of nulled votes (amount all types of nulled votes);
  - **null_votes**: number of null votes (vote made by a voter);
  - **technical_null_votes**: number of votes which got nulled due to some issue (disqualified candidate, cancelled polling station and so on);
  - **valid_votes**: number of valid notes (used to elect a candidate);
  - **nulled_votes**: number of votes which got nulled;
  - **judically_nulled_votes**: number of votes which got nulled by justice;
  - **vote_count**: total number of votes, that is, vote_count = blank_votes + total_null_votes + valid_votes;
  - **candidate_number**: unique number that identifies the candidate;
  - **candidate_vote_count**: total number of votes given for a candidate;
  - **legend_votes**: votes given to a party.

# Exploratory Data Analysis

## Univariate Analysis
We start our analysis by checking the numerical data by providing
basic statistics about its central tendency and dispersion.

In [ ]:
data_frame.describe().T

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
## generate the histograms for each numerical column in the data frame
numeric_cols = data_frame.select_dtypes(include='number').columns

## normalize the data in order to have the histograms comparable
min_max_scaler = MinMaxScaler()
normalized_data = min_max_scaler.fit_transform(data_frame[numeric_cols])
normalized_df = pd.DataFrame(normalized_data,
                             columns=numeric_cols)

## creates a figure with 3 histograms for each row
num_numeric_cols = len(numeric_cols)
fig, axs = plt.subplots(num_numeric_cols // 3,
                        3,
                        figsize=(18,14))

## the axes was flatten to make easier to iterate over it
axs = axs.flatten()

## draw the histograms
for i, column in enumerate(normalized_df):
    normalized_df[column].hist(ax=axs[i],
                               edgecolor='white',
                               log=True)
    axs[i].set_title(column)
    axs[i].set_xlabel(f'Number of {column}(s)')
    axs[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

The histograms of variables related to the vote are asymmetric to the
right and their pattern are a bit similar. Note that there is a high
concentration of data at the firsts bins, which could be happening for
a few reasons. First, we are counting the votes for mayors with
councilors. Since the amount of former job is greater than mayors
and usually councilors are elected with less amount of votes, then
this could explain the data concentration at the firsts bins. Moreover,
these histograms also put together data from cities with different
amount of population, which also affects the number of votes. And also,
this could be the reason of some gaps (which suggests existence of
    outliers) in some histograms. Due to these observations, in our
multivariate analysis, we will consider breaking down voting data by
job, state and party to have a better understating of these data.

In [ ]:
## showing the skewness for each numeric variable, in order to
## check how data is assymatric (or not)
normalized_df.skew()

Capital distribution is expected to have few frequencies of 1, since
the minority of cities are capitals.

It is important to recall that, mayor and councilor number have a
prefix which identifies their main party. Since the prefix and party
number are the same, then we will only analyze main party data
(categorical variable).

Below we plot the box plot for each numeric variable. They confirmed the
existence of outliers. As mentioned before, this can happen because a
councilor requires less votes to be elected than a mayor. And also, we
will how candidate vote is distributed among the jobs, because it could
happen that a few candidates concentrate the majority of votes. In
addition, we will check the relation between votes, job and state, since
population number may affect the number of votes.

In [ ]:
## creates a figure with 3 histograms for each row
fig, axs = plt.subplots(num_numeric_cols // 3,
                        3,
                        figsize=(16,14))

## the axes was flatten to make easier to iterate over it
axs = axs.flatten()

## draw the boxplots
for i,column in enumerate(numeric_cols):
    axs[i].boxplot(normalized_df[column])
    axs[i].set_title(column)

plt.tight_layout()
plt.show()

In [ ]:
## obtaining the categorical variables of the data frame
category_cols = data_frame.select_dtypes(include='object').columns

## priting the amount of unique categories for each
## categoriacal variable
for i, column in enumerate(category_cols):
    print(f'Data for the column: {column}\n'
          f'{data_frame[column].value_counts()}\n\n')

For the variable nome_municipio, we can note that highest frequencies are
associated with the cities with high population. Which is a expected
result since the number of candidates tends to be higher in big cities, as
the number of councilors depends on the number of citizens. The same
happens to uf, which identifies a Brazilian state.

We will not analyze further candidate_name and candidate_vice_name
since they are redundant information, because a candidate is uniquely
identified by a number. Moreover, it is more convenient to work with
numbers. In particular, for candidate_vice_name, you can see that
exists 500035 blank names (null). This happens because councilors
do not have a vice.

Regarding the job, as expected, the majority of candidates are
councilors, whereas the minority are mayors. And most of votes are
valid, just a few of them are null.

The candidate_coligation shows a candidate electoral alliance, however,
for voting purposes, what matter is the candidate main party. So we
will focus our analysis on this column. By executing the method
*value counts* of this column, we can see that the first 5 parties are
have center or right political alignment. Below we plot bar plots for
some variable in order to better visualize their data.

In [ ]:
## creating bar plots for some categorical variables
columns = ['uf', 'candidate_vote_destination', 'job', 'main_party']

fig, axs = plt.subplots(2,
                        2,
                        figsize=(16,16))

axs = axs.flatten()

for i,column in enumerate(columns):
    counts = data_frame[column].value_counts()
    axs[i].bar(counts.index,
               counts.values,
               log=True)
    axs[i].set_title(column)

    ## rotating the labels for the main_party column
    ## since there are too many labels
    if column == 'main_party' :
        axs[i].tick_params(labelrotation=90)

plt.tight_layout()
plt.show()

### Multivariate Analysis
We start our multivariate analysis by checking the linear
correlation between numerical variables.

In [ ]:
import seaborn as sns

## correlação entre as variáveis numéricas
correlation_matrix = data_frame[numeric_cols].corr()
sns.heatmap(correlation_matrix)
correlation_matrix

From the correlation matrix, we have the following notes:

- the variables *elector_presence*, *absentees*, *nominal_votes*,
  *blank_votes*, *total_null_votes*, *null_votes*, *valid_votes*,
  *judically_nulled_votes*, *vote_count* and *legend_votes* are strongly
  linear correlated to each other. For example, if the
  number of elector presence increases, then the absentees the votes
  (nominal, null, total and count) increases as well; which is expected;

- nulled_votes, candidate_number and candidate_vote_count do not have a
  strong linear correlation to any other variable;

- technical_null_votes and capital have a quite, but not strong, linear
  correlation to the other variables.

Now we are going to check how the vote is related to state and the job.

In [ ]:
## creates the figure and axes objects for plotting
fig, eixo = plt.subplots(1,2, figsize=(12,10))

## compute the sum of candidate votes grouped per job
total_votes_per_job = data_frame.groupby(by='job')['candidate_vote_count'].sum()
total_votes_per_job = total_votes_per_job.reset_index(['job'])

g = sns.barplot(data=total_votes_per_job,
                x='job',
                y='candidate_vote_count',
                ax=eixo[0]
)

g.set_title('Total votes per job.')

## computes the average of candidate votes grouped per job
total_votes_per_job = data_frame.groupby(by='job')['candidate_vote_count'].mean()
total_votes_per_job = total_votes_per_job.reset_index(['job'])

h = sns.barplot(
    data=total_votes_per_job,
    x='job',
    y='candidate_vote_count',
    ax=eixo[1]
)

h.set_title('Average votes per job.')

plt.show()

In [ ]:
## code to plot bar graphs with total (resp. average) the candidate
## votes per job and state
fig, eixo = plt.subplots(1,2, figsize=(12,12))

vote_per_job = data_frame.groupby(by=['job', 'uf'])['candidate_vote_count'].sum()
vote_per_job = vote_per_job.reset_index(['job', 'uf'])

g = sns.barplot(
    data=vote_per_job,
    x='candidate_vote_count',
    y='uf',
    hue='job',
    ax=eixo[0]
)

g.set_title('Sum of candidate votes per job and state.')

vote_per_job = data_frame.groupby(by=['job', 'uf'])['candidate_vote_count'].mean()
vote_per_job = vote_per_job.reset_index(['job', 'uf'])

h = sns.barplot(
    data=vote_per_job,
    x='candidate_vote_count',
    y='uf',
    hue='job',
    ax=eixo[1]
)

h.set_title('Average of candidate votes per job and state.')

plt.show()

In [ ]:
## checking how voting data is distributed for councilors per state
states = data_frame['uf'].unique()

for state in states:
    print(f'Basic statistics for councilors for state {state}.')
    councilors = data_frame[
    (data_frame['uf'] == state) & (data_frame['job'] == 'vereador')
    ]
    print(councilors['candidate_vote_count'].describe(percentiles=[.1, .25, .5, .75, .9]).T,
          end='\n\n')

## ploting a histogram of candidate votes for TO state
## to visualize its distribution
## TO state was picked because it generates a better visualization of
## the data distribution and it also has a distribution similar to all
## other states
fig, ax = plt.subplots()

councilors = data_frame[
(data_frame['uf'] == 'TO') & (data_frame['job'] == 'vereador')
]

councilors['candidate_vote_count'].hist(ax=ax)
ax.set_title(f'Histogram of candidate votes for councilors at TO')
ax.set_xlabel('Amount of candidate votes')
ax.set_ylabel('Frequency')
plt.show()

In [ ]:
## checking show voting data is distributed for mayors per state
states = data_frame['uf'].unique()

for state in states:
    print(f'Basic statistics for mayors for state {state}.')
    mayors = data_frame[
    (data_frame['uf'] == state) & (data_frame['job'] == 'prefeito')
    ]
    print(mayors['candidate_vote_count'].describe(percentiles=[.1, .25, .5, .75, .9]).T,
          end='\n\n')

fig, ax = plt.subplots()
mayors['candidate_vote_count'].hist(ax=ax)
ax.set_title(f'Histogram of candidate votes for mayors at {state}')
ax.set_xlabel('Amount of candidate votes')
ax.set_ylabel('Frequency')
plt.show()

From the graphs above, we can note perceptible differences between jobs
when average candidate vote is analyzed, whereas total votes do not show a
noticeable difference. When we breakdown these votes per state, the
average votes between mayors and councilors show a great
difference. Average votes for councilors do not show a great variance and
they are quite similar, while average mayor votes have a greater variance.

Recall that we have 500035 councilors and 18640 mayors and, respectively,
they had 97261711 and 102463947 votes in total. Since the number of
councilors is greater than the number of mayors, then in average
councilors received less amount of votes than mayors. This allows us
to understand why we had a high votes frequency are at the first bins
in the histograms for numerical univariate analysis.

Moreover, we expected the high populated states had a great impact on the
number of votes received by a candidate. It does when we take in
consideration only the total votes. However, councilors average candidate
votes are quite similar among states (uf), and the mayor average votes for
Pernambuco (PE) and São Paulo (SP) are close, however Pernambuco has
7,018,098 voters and São Paulo 34,667,793 voters. When analyzing candidate
vote histogram for both jobs, we can note that only a few candidates
received a high amount of votes and most of them received a small amount
of votes. And also, 75% of data is within distance 0 from the
mean. Therefore, the job has a greater influence in candidate vote than
the state.

We also created a [dashboard](https://public.tableau.com/app/profile/gabriel.sobral/viz/election_analysis_17428265010000/Votebashboard)
with candidate vote per state and candidate vote per party, which can
be filtered by state, to provide a better data visualization for these
variables. In docs directory, you can find the *Tableau* and *Power BI*
files used to generate this dashboard.

In [ ]:
fig, eixo = plt.subplots(2, 1, figsize=(16, 22))

eixo = eixo.flatten()

vote_per_party = data_frame.groupby(by=['job', 'main_party'])['candidate_vote_count'].sum()
vote_per_party = vote_per_party.reset_index()

g = sns.barplot(
    data=vote_per_party,
    x='candidate_vote_count',
    y='main_party',
    hue='job',
    ax=eixo[0]
)

g.set_title('Total candidates votes per job and party (main).')

vote_per_party = data_frame.groupby(by=['job', 'main_party'])['candidate_vote_count'].mean()
vote_per_party = vote_per_party.reset_index()

h = sns.barplot(
    data=vote_per_party,
    x='candidate_vote_count',
    y='main_party',
    hue='job',
    ax=eixo[1]
)

h.set_title('Average candidates votes per job and party (main).')

plt.show()

For some main parties considered small in Brazil (AVANTE, CIDADANIA, DC
and other), the total votes for councilors are greater than mayors. Since
the campaign for councilors are usually cheaper than mayors, then this
could explain why we found this result. However, in average number of
votes, mayors received many more votes than councilors. These graphs show
that the job of mayor tend to receive less votes than councilors when the
main party is small, in contrast, when the party is big, the situation
tends to be the opposite.

The latest graphs allowed us to understand more about the relation between
party, state and vote. Now we are going to check some data about elected
mayors and councilors.

In [ ]:
## Analyzing the number of mayors elected by main party
query_mayors = data_frame[
    (data_frame['job'] == 'prefeito') &
    (data_frame['elector_count'] == 's')
]

mayors_analysis = query_mayors.groupby(by=['main_party']).agg(
    amount=('candidate_vote_count', 'count')
)

number_elected_mayors = mayors_analysis['amount'].sum()
mayors_analysis['%'] = mayors_analysis['amount'] / number_elected_mayors * 100
mayors_analysis['%'] = round(mayors_analysis['%'], 2)
mayors_analysis.sort_values('amount', inplace=True, ascending=False)

## plotting the graphic with the analyzes mayors_analysis
## creating a pallet of colors using seaborn
color_palette = sns.color_palette('magma', len(mayors_analysis))

## configuring the size of the graph
plt.figure(figsize=(20,6))

## plotting the graph itself
plt.bar(mayors_analysis.index,
        mayors_analysis['amount'],
        width=0.9,
        color=color_palette)

## configuring the title
plt.title('Mayors elected in Brazil',
          loc='left',
          fontsize=20,
          color='#404040',
          fontweight=600)

## configuring the labels
plt.ylabel('Amount of mayors')
plt.xlabel('Parties')
plt.xticks(rotation=90)

## changing y axis limit value to add some space after the
## maximum value
plt.ylim(0, mayors_analysis['amount'].max() * 1.1)

## adding labels for each bar of the graph
for position, value in enumerate(mayors_analysis['amount']):
    plt.text(
        ## position of the label
        position -0.3,
        value +10,
        ## label text
        value,
        ## label color
        color=color_palette[position],
        ## label size and font
        size=12,
        fontweight=700
    )

plt.annotate(
    f'Elected in Brazil: {mayors_analysis["amount"].sum()}',
    xy=(0.99,0.94),
    xycoords='axes fraction',
    ha='right',
    va='center',
    color='green',
    fontsize=14,
    bbox=dict(facecolor='#ffffff',
               edgecolor='green',
               boxstyle='round',
               pad=0.25)
)

In [ ]:
## analyzing data for councilors
query_councilors = data_frame[
    (data_frame['job'] == 'vereador') &
    (data_frame['elector_count'] == 's')
]

councilors_analysis = query_councilors.groupby(by='main_party').agg(
    amount = ('candidate_vote_count', 'count')
)

number_elected_councilors = councilors_analysis['amount'].sum()
councilors_analysis['%'] = councilors_analysis['amount'] / number_elected_councilors * 100
councilors_analysis['%'] = round(councilors_analysis['%'], 2)
councilors_analysis.sort_values('amount', inplace=True, ascending=False)

plt.figure(figsize=(12,10))
plt.hlines(
    y=councilors_analysis.index,
    xmin=0,
    xmax=councilors_analysis['amount'],
    lw=5,
    color=color_palette,
    alpha=0.5
)

plt.scatter(
    councilors_analysis['amount'],
    councilors_analysis.index,
    s=100,
    color=color_palette,
    alpha=0.8
)

plt.title('Councilors Elected in Brazil',
          loc='left',
          fontsize=20,
          color='#404040',
          fontweight=500)

plt.annotate(
    f'Elected in Brazil: {councilors_analysis["amount"].sum()}',
    xy=(0.99,0.94),
    xycoords='axes fraction',
    ha='right',
    va='center',
    color='green',
    fontsize=14,
    bbox=dict(facecolor='#ffffff',
               edgecolor='green',
               boxstyle='round',
               pad=0.25)
)

From the last two graphs, we can note the representative of the main parties
for councilors and mayors. Moreover, we can tell which party can be
considered big in Brazil by the number of elected mayors or
councilors. For example, we can consider the first five parties (for both
jobs) big.

For the last, we will check if there is any relationship between the
number of mayors and councilors elected by party.

In [ ]:
## correlation analysis
## checking if there is any relationship between the number of mayors
## and councilors elected by party
correlation_table = mayors_analysis['amount'].reset_index()
correlation_table = pd.merge(correlation_table,
                             councilors_analysis.reset_index(),
                             on=['main_party'],
                             how='inner')
correlation_table.columns = ['Party', 'Mayors', 'Councilors', '%']
correlation_table.drop(columns=['%'], inplace=True)

sns.regplot(
    x=correlation_table['Mayors'],
    y=correlation_table['Councilors'],
    ci=95,
    scatter_kws={
        'color': 'blue',
        's': 80,
        'alpha': 0.5},
    line_kws={
        'color': 'orange',
        'alpha': 0.2,
        'lw': 2
    }
)

plt.title('Mayors vs Councilors elected')

## adding the party for mayors and councilors
for line in range(0, correlation_table.shape[0]):
    plt.text(
        correlation_table['Mayors'][line] + 0.8,
        correlation_table['Councilors'][line],
        correlation_table['Party'][line],
        size='medium',
        color='gray',
        weight='semibold'
    )

We can see that there is postive correlantion between the number of elected
councilors and mayors, indicating that bigger parties tend to elect more
cancidates.

# Conclusion

For the election dataset from 2022, we could describe its categorical and
numerical data. About candidate vote data, we found they are highly
concentrated at first bins of histograms and are asymmetric to right. We
further analyzed them and found the candidate vote is highly influenced by
its job. The state important when we consider the total candidate votes,
but in average its importance is reduced. For both jobs, we found a few
candidates receive a high amount of votes, whereas most of candidates do
not. These results explain why the data on histograms are concentrated at
the firsts bins. We and found a positive correlation between the number of
councilors and mayors elected by party. Our exploratory data analysis
could provide some insight about the data and it could be extended by
analyzing more previous data.